# Bench Press vs Shoulder Press vs Lateral Raise — IMU pattern comparison

Compares wrist-worn IMU signal **patterns** (not just amplitude) between
three RecoFit exercises: **Chest Press (rack)**, **Squat Rack Shoulder
Press**, and **Lateral Raise**.

**Key idea:** accel *magnitude* (`sqrt(x^2+y^2+z^2)`) throws away
direction — it tells you how hard you moved, not which way. The three
exercises move in different planes (forward push, vertical push, lateral
raise), so the per-axis signal is where the real exercise-specific
pattern lives. This notebook looks at magnitude first (baseline), then
per-axis.

**Data required:** `data/recordings.csv` and `data/samples.csv`.

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "../data"

recordings = pd.read_csv(f"{DATA_DIR}/recordings.csv")
samples = pd.read_csv(f"{DATA_DIR}/samples.csv")

print(recordings.shape, samples.shape)
recordings.head()

## 2. Select the three exercises to compare

In [ ]:
BENCH = "Chest Press (rack)"
SHOULDER = "Squat Rack Shoulder Press"
LATERAL = "Lateral Raise"
EXERCISES = [BENCH, SHOULDER, LATERAL]

recordings["activity_name"].value_counts()

In [ ]:
recordings_by_exercise = {ex: recordings[recordings["activity_name"] == ex] for ex in EXERCISES}

for ex, recs in recordings_by_exercise.items():
    print(f"{ex}: {len(recs)} recordings")

## 3. Example recording per exercise — accel magnitude (baseline)

Accelerometer magnitude over time, one example set per exercise. This is
the amplitude-only view — useful as a baseline, but it can't tell us
*which direction* the arm moved.

In [ ]:
def load_sample_signal(record_uid: str) -> pd.DataFrame:
    rec = samples[samples["record_uid"] == record_uid].sort_values("sample_index")
    mag = np.sqrt(rec["accel_x_g"]**2 + rec["accel_y_g"]**2 + rec["accel_z_g"]**2)
    return pd.DataFrame({
        "time_s": rec["time_s"].to_numpy(),
        "accel_x_g": rec["accel_x_g"].to_numpy(),
        "accel_y_g": rec["accel_y_g"].to_numpy(),
        "accel_z_g": rec["accel_z_g"].to_numpy(),
        "accel_mag_g": mag.to_numpy(),
    })

example_uid = {ex: recordings_by_exercise[ex].iloc[0]["record_uid"] for ex in EXERCISES}
example_signal = {ex: load_sample_signal(uid) for ex, uid in example_uid.items()}
example_uid

In [ ]:
fig, axes = plt.subplots(len(EXERCISES), 1, figsize=(10, 8), sharex=False)

for ax, ex in zip(axes, EXERCISES):
    sig = example_signal[ex]
    ax.plot(sig["time_s"], sig["accel_mag_g"])
    ax.set_title(f"{ex} — {example_uid[ex]}")
    ax.set_ylabel("|accel| (g)")

axes[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

## 4. Per-axis waveform — where the real pattern is

Same three recordings, but now split into the x/y/z axes instead of
collapsed into magnitude. Rows = exercise, columns = axis. Look for:
which axis carries most of the movement (a large-amplitude axis vs a
near-flat one), and whether that dominant axis differs by exercise —
that's the movement-plane signature magnitude erases.

In [ ]:
axis_cols = ["accel_x_g", "accel_y_g", "accel_z_g"]

fig, axes = plt.subplots(len(EXERCISES), len(axis_cols), figsize=(13, 8), sharex=False)

for row, ex in enumerate(EXERCISES):
    sig = example_signal[ex]
    for col, axis_col in enumerate(axis_cols):
        ax = axes[row, col]
        ax.plot(sig["time_s"], sig[axis_col], linewidth=1)
        if row == 0:
            ax.set_title(axis_col)
        if col == 0:
            ax.set_ylabel(ex, fontsize=9)

plt.tight_layout()
plt.show()

## 5. Per-recording summary stats — axis-aware features

For every recording: magnitude stats (as before) *plus* per-axis mean,
std, and range for x/y/z. The per-axis features are what should actually
separate these exercises, since they encode movement direction, not
just movement intensity.

In [ ]:
def summarize(record_uid: str, activity_reps, master_sample_rows) -> dict:
    sig = load_sample_signal(record_uid)
    out = {
        "record_uid": record_uid,
        "activity_reps": activity_reps,
        "master_sample_rows": master_sample_rows,
        "duration_s": sig["time_s"].max() - sig["time_s"].min() if len(sig) else np.nan,
        "accel_mag_mean_g": sig["accel_mag_g"].mean(),
        "accel_mag_max_g": sig["accel_mag_g"].max(),
        "accel_mag_std_g": sig["accel_mag_g"].std(),
    }
    for axis_col in ["accel_x_g", "accel_y_g", "accel_z_g"]:
        out[f"{axis_col}_mean"] = sig[axis_col].mean()
        out[f"{axis_col}_std"] = sig[axis_col].std()
        out[f"{axis_col}_range"] = sig[axis_col].max() - sig[axis_col].min()
    return out

def summary_table(recs: pd.DataFrame) -> pd.DataFrame:
    rows = [
        summarize(r.record_uid, r.activity_reps, r.master_sample_rows)
        for r in recs.itertuples()
    ]
    return pd.DataFrame(rows)

summary_by_exercise = {ex: summary_table(recordings_by_exercise[ex]) for ex in EXERCISES}
summary_by_exercise[BENCH].describe()

## 6. Side-by-side comparison

First the old magnitude-only view (baseline), then the per-axis means —
compare how much cleaner the separation looks once direction is kept.

In [ ]:
compare = pd.concat([
    summary_by_exercise[ex].assign(exercise=ex) for ex in EXERCISES
])

compare.boxplot(column="accel_mag_max_g", by="exercise", figsize=(6, 5))
plt.title("Peak accel magnitude per set (amplitude only — baseline)")
plt.suptitle("")
plt.ylabel("|accel| max (g)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, axis_col in zip(axes, ["accel_x_g_mean", "accel_y_g_mean", "accel_z_g_mean"]):
    compare.boxplot(column=axis_col, by="exercise", ax=ax)
    ax.set_title(axis_col)
    ax.set_xlabel("")

plt.suptitle("Per-axis mean accel by exercise — direction, not just intensity")
plt.tight_layout()
plt.show()

## Next steps

- Do the same per-axis comparison for gyroscope (`gyro_x_dps` etc.) —
  rotation direction is likely just as diagnostic as accel direction.
- Segment individual reps within a set (using `activity_reps`) and
  compare rep-level waveform shape, not just whole-set summary stats.
- Train an actual classifier (e.g. decision tree or logistic regression)
  on the per-axis features from section 5, with a **subject-wise**
  train/test split (never split by row — the same person's sets must
  not appear in both train and test), and report a confusion matrix.
  Only that number tells you whether this is actually separable — box
  plots are a hint, not a result.

## 7. Animated rep pattern — "from the side"

**Honesty check first:** accelerometer data alone cannot be turned into a
real spatial trajectory — double-integrating acceleration to get position
drifts wildly within a second or two, it would be fabricated, not
measured. So this is *not* a physical reconstruction of the arm's path.

What it *is*: for each exercise, take the same example recording used
above, split it into its individual reps, resample and **average all
reps into one canonical cycle** (an average bottom point, an average top
point, and everything in between, consistently connected), then animate
a marker looping through that cycle. All three exercises side by side —
watch the shape and rhythm, not absolute values.

In [ ]:
def dominant_axis_signal(record_uid: str):
    """Return (time, signal, axis_name) for whichever accel axis has the
    largest range in this recording — the axis carrying the movement."""
    sig = load_sample_signal(record_uid)
    axis_cols = ["accel_x_g", "accel_y_g", "accel_z_g"]
    ranges = {c: sig[c].max() - sig[c].min() for c in axis_cols}
    best_axis = max(ranges, key=ranges.get)
    return sig["time_s"].to_numpy(), sig[best_axis].to_numpy(), best_axis

def average_rep_cycle(record_uid: str, n_reps, n_points: int = 60):
    """Split the dominant-axis signal into n_reps equal-duration chunks,
    resample each to n_points, and average them into one canonical
    bottom-to-top-to-bottom cycle, normalized to [0, 1]."""
    t, y, axis_name = dominant_axis_signal(record_uid)
    n_reps = int(n_reps) if n_reps and n_reps > 0 else 10  # fallback if reps is missing/-1

    edges = np.linspace(t.min(), t.max(), n_reps + 1)
    cycles = []
    for i in range(n_reps):
        mask = (t >= edges[i]) & (t <= edges[i + 1])
        if mask.sum() < 4:
            continue
        seg_t_norm = np.linspace(0, 1, mask.sum())
        resampled = np.interp(np.linspace(0, 1, n_points), seg_t_norm, y[mask])
        cycles.append(resampled)

    canonical = np.mean(cycles, axis=0)
    canonical_norm = (canonical - canonical.min()) / (canonical.max() - canonical.min())
    return canonical_norm, axis_name

In [ ]:
canonical_cycles = {}
for ex in EXERCISES:
    uid = example_uid[ex]
    reps = recordings.loc[recordings["record_uid"] == uid, "activity_reps"].iloc[0]
    cycle, axis_name = average_rep_cycle(uid, reps)
    canonical_cycles[ex] = {"cycle": cycle, "axis": axis_name, "reps": reps}
    print(f"{ex}: dominant axis = {axis_name}, reps used = {reps}")

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_points = next(iter(canonical_cycles.values()))["cycle"].shape[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 5))

traces, markers = [], []
for ax, ex in zip(axes, EXERCISES):
    ax.set_xlim(-0.5, 0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.axhline(1, color="grey", linestyle="--", linewidth=1)
    ax.set_title(f"{ex}\n(axis: {canonical_cycles[ex]['axis']})", fontsize=9)
    ax.set_xticks([])
    ax.set_ylabel("normalized position (0=avg bottom, 1=avg top)")
    (trace_line,) = ax.plot([], [], color="tab:blue", alpha=0.5, linewidth=2)
    (marker_dot,) = ax.plot([], [], "o", color="tab:red", markersize=14)
    traces.append(trace_line)
    markers.append(marker_dot)

def init():
    for trace_line, marker_dot in zip(traces, markers):
        trace_line.set_data([], [])
        marker_dot.set_data([], [])
    return traces + markers

def update(frame):
    idx = frame % n_points
    for trace_line, marker_dot, ex in zip(traces, markers, EXERCISES):
        cycle = canonical_cycles[ex]["cycle"]
        trace_line.set_data(np.zeros(idx + 1), cycle[: idx + 1])
        marker_dot.set_data([0], [cycle[idx]])
    return traces + markers

anim = FuncAnimation(fig, update, init_func=init, frames=n_points, interval=60, blit=True, repeat=True)
plt.tight_layout()
plt.close(fig)
HTML(anim.to_jshtml())

## 8. Gyroscope — per-axis pattern

We've only looked at the accelerometer so far. The sensor also reports
gyroscope (rotation rate, deg/s) on the same 3 axes — not shown yet.

One IMU-placement detail worth knowing (from the RecoFit paper itself):
the sensor is worn on the forearm, and only its **X axis is guaranteed
to point along the arm** — Y and Z can end up pointing anywhere in the
plane perpendicular to the arm, because the band can rotate around the
arm between sessions/subjects. That's exactly why section 4 and section
7 auto-detect the dominant axis per recording instead of assuming a
fixed one — with gyro we'll do the same rather than assume, say,
`gyro_x` is always the meaningful one.

Same three example recordings as sections 3-4-7, so this is directly
comparable to what we already saw for accel.

In [ ]:
def load_gyro_signal(record_uid: str) -> pd.DataFrame:
    rec = samples[samples["record_uid"] == record_uid].sort_values("sample_index")
    return pd.DataFrame({
        "time_s": rec["time_s"].to_numpy(),
        "gyro_x_dps": rec["gyro_x_dps"].to_numpy(),
        "gyro_y_dps": rec["gyro_y_dps"].to_numpy(),
        "gyro_z_dps": rec["gyro_z_dps"].to_numpy(),
    })

example_gyro_signal = {ex: load_gyro_signal(uid) for ex, uid in example_uid.items()}

gyro_axis_cols = ["gyro_x_dps", "gyro_y_dps", "gyro_z_dps"]
fig, axes = plt.subplots(len(EXERCISES), len(gyro_axis_cols), figsize=(13, 8), sharex=False)

for row, ex in enumerate(EXERCISES):
    sig = example_gyro_signal[ex]
    for col, axis_col in enumerate(gyro_axis_cols):
        ax = axes[row, col]
        ax.plot(sig["time_s"], sig[axis_col], linewidth=1, color="tab:green")
        if row == 0:
            ax.set_title(axis_col)
        if col == 0:
            ax.set_ylabel(ex, fontsize=9)

plt.tight_layout()
plt.show()

## 9. Does gyro separate Lateral Raise from the presses more cleanly?

Hypothesis: Lateral Raise rotates the arm out to the side — a distinct
rotation plane from either press. If true, gyro should show a much
cleaner separation than accel did for the hard pair (Chest Press vs
Shoulder Press), because those two are close in *translation* but may
still differ in *rotation*.

One thing to get right: unlike accel (which has a constant gravity
offset we could just average), gyro oscillates around 0 within a rep —
you rotate one way, then rotate back. The **mean cancels out to ~0
regardless of exercise**, so it's not a useful feature here. What *is*
useful: **std** (how much rotation is happening on that axis, magnitude
regardless of direction).

In [ ]:
def gyro_axis_stats(record_uid: str) -> dict:
    sig = load_gyro_signal(record_uid)
    out = {"record_uid": record_uid}
    for axis_col in gyro_axis_cols:
        out[f"{axis_col}_std"] = sig[axis_col].std()
    return out

def gyro_summary_table(recs: pd.DataFrame) -> pd.DataFrame:
    rows = [gyro_axis_stats(uid) for uid in recs["record_uid"]]
    return pd.DataFrame(rows)

gyro_summary_by_exercise = {ex: gyro_summary_table(recordings_by_exercise[ex]) for ex in EXERCISES}

compare_gyro = pd.concat([
    gyro_summary_by_exercise[ex].assign(exercise=ex) for ex in EXERCISES
])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, axis_col in zip(axes, [f"{c}_std" for c in gyro_axis_cols]):
    compare_gyro.boxplot(column=axis_col, by="exercise", ax=ax)
    ax.set_title(axis_col)
    ax.set_xlabel("")

plt.suptitle("Per-axis gyro std (how much rotation) by exercise")
plt.tight_layout()
plt.show()

## 10. Literature method (RecoFit-style) vs. ours — side by side

Everything above (sections 1-9) is **our method**: raw mean / std /
range, and a crude "pick the single raw axis with the biggest range"
for the dominant axis.

This section implements a simplified version of what the RecoFit paper
actually uses for its recognition stage — labeled **LITERATURE** in
every cell/plot below so it's never confused with **OURS**:

- **PCA-based axis projection** (`aYZPC1`, `gPC1`) instead of picking one
  raw axis — finds the best *combination* of axes, robust to how the
  band happens to be rotated on the arm (paper's "dimensionality
  reduction" lesson).
- **RMS** instead of mean/std — same spirit as our gyro-std trick, but
  the standard, named version of it.
- **Autocorrelation-based tempo** — a feature we had *zero* version of:
  how long one rep takes, independent of amplitude or direction.

At the end we put our best features and the literature features
side by side on the hard pair (**Chest Press vs Shoulder Press**) and
score them, so the difference isn't just "eyeballed from a plot" but a
number.

### 10.1 PCA-based axis projection — LITERATURE

In [ ]:
def pca_first_component(X: np.ndarray) -> np.ndarray:
    """Project a multi-axis signal (n_samples, n_axes) onto its first
    principal component: the single direction (a combination of the
    input axes) along which the signal varies the most. This is what
    the RecoFit paper calls aYZPC1 / gPC1 -- unlike our section 4/7
    'pick whichever raw axis has the biggest range', PCA doesn't throw
    away the other axes, it blends them optimally. No sklearn needed,
    plain SVD."""
    Xc = X - X.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[0]

def literature_signals(record_uid: str) -> dict:
    """The 3 signals RecoFit's recognition stage uses: aX (arm-axis
    accel, the one axis with fixed physical meaning), aYZPC1 (PCA of the
    other two accel axes), gPC1 (PCA of all 3 gyro axes)."""
    sig = samples[samples["record_uid"] == record_uid].sort_values("sample_index")
    aX = sig["accel_x_g"].to_numpy()
    aYZPC1 = pca_first_component(sig[["accel_y_g", "accel_z_g"]].to_numpy())
    gPC1 = pca_first_component(sig[["gyro_x_dps", "gyro_y_dps", "gyro_z_dps"]].to_numpy())
    return {"aX": aX, "aYZPC1": aYZPC1, "gPC1": gPC1}

### 10.2 RMS and autocorrelation tempo — LITERATURE

In [ ]:
def rms(x: np.ndarray) -> float:
    """Root-mean-square amplitude -- like our gyro std trick, but the
    named, standard version (doesn't require the signal to be
    zero-mean, unlike std)."""
    return float(np.sqrt(np.mean(x**2)))

def autocorr_tempo_feature(x: np.ndarray, fs: float = 50.0) -> float:
    """Simplified version of RecoFit's autocorrelation feature: the lag
    (in seconds) of the strongest non-trivial self-similarity peak --
    roughly 'how long one rep takes'. FFT-based so it stays fast even on
    our longer recordings. This is a feature ours (mean/std/RMS) cannot
    see at all: two exercises with identical amplitude but different
    tempo look the same to mean/std/RMS, but not to this."""
    x = x - x.mean()
    n = len(x)
    if n < 20:
        return np.nan
    size = 1
    while size < 2 * n:
        size *= 2
    fx = np.fft.fft(x, size)
    acf = np.fft.ifft(fx * np.conjugate(fx)).real[:n]
    if acf[0] == 0:
        return np.nan
    acf = acf / acf[0]
    min_lag = int(0.2 * fs)  # ignore lags under 0.2s -- not a real rep
    if min_lag >= n - 1:
        return np.nan
    peak_lag = int(np.argmax(acf[min_lag:])) + min_lag
    return peak_lag / fs

### 10.3 Compute LITERATURE features for every recording

In [ ]:
def literature_features(record_uid: str) -> dict:
    sigs = literature_signals(record_uid)
    out = {"record_uid": record_uid}
    for name, x in sigs.items():
        out[f"{name}_rms"] = rms(x)
        out[f"{name}_tempo_s"] = autocorr_tempo_feature(x)
    return out

def literature_summary_table(recs: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([literature_features(uid) for uid in recs["record_uid"]])

literature_by_exercise = {ex: literature_summary_table(recordings_by_exercise[ex]) for ex in EXERCISES}
compare_literature = pd.concat([
    literature_by_exercise[ex].assign(exercise=ex) for ex in EXERCISES
])
compare_literature.describe()

### 10.4 OURS vs LITERATURE — side by side, visually

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

compare.boxplot(column="accel_x_g_mean", by="exercise", ax=axes[0])
axes[0].set_title("OURS: accel_x mean")
axes[0].set_xlabel("")

compare_gyro.boxplot(column="gyro_z_dps_std", by="exercise", ax=axes[1])
axes[1].set_title("OURS: gyro_z std")
axes[1].set_xlabel("")

compare_literature.boxplot(column="aYZPC1_rms", by="exercise", ax=axes[2])
axes[2].set_title("LITERATURE: aYZPC1 RMS")
axes[2].set_xlabel("")

compare_literature.boxplot(column="aX_tempo_s", by="exercise", ax=axes[3])
axes[3].set_title("LITERATURE: aX tempo (s)")
axes[3].set_xlabel("")

plt.suptitle("Ours vs. literature-style features, all three exercises")
plt.tight_layout()
plt.show()

### 10.5 The actual test: OURS vs LITERATURE on the hard pair (Chest Press vs Shoulder Press)

A plot can look convincing either way depending on how you squint at it.
Score it instead: for each feature, `|median(A) - median(B)| /
(IQR(A) + IQR(B))` -- a coarse separation score. Higher = the two
exercises' distributions overlap less on that feature. This is *not* a
trained classifier accuracy, just a fast, honest way to rank features
before spending time on a real model (see the Next steps note in
section 6 about doing that properly, with a subject-wise split).

In [ ]:
def separation_score(df: pd.DataFrame, feature_col: str, ex_a: str, ex_b: str) -> float:
    a = df.loc[df["exercise"] == ex_a, feature_col].dropna()
    b = df.loc[df["exercise"] == ex_b, feature_col].dropna()
    med_gap = abs(a.median() - b.median())
    spread = (a.quantile(0.75) - a.quantile(0.25)) + (b.quantile(0.75) - b.quantile(0.25))
    return med_gap / spread if spread > 0 else np.nan

ours_candidates = {
    "accel_x_g_mean": compare,
    "accel_y_g_mean": compare,
    "accel_z_g_mean": compare,
    "accel_mag_max_g": compare,
    "gyro_x_dps_std": compare_gyro,
    "gyro_y_dps_std": compare_gyro,
    "gyro_z_dps_std": compare_gyro,
}
literature_candidates = {
    "aX_rms": compare_literature,
    "aYZPC1_rms": compare_literature,
    "gPC1_rms": compare_literature,
    "aX_tempo_s": compare_literature,
    "aYZPC1_tempo_s": compare_literature,
    "gPC1_tempo_s": compare_literature,
}

rows = []
for col, df in ours_candidates.items():
    rows.append({"method": "OURS", "feature": col,
                 "chest_vs_shoulder_separation": separation_score(df, col, BENCH, SHOULDER)})
for col, df in literature_candidates.items():
    rows.append({"method": "LITERATURE", "feature": col,
                 "chest_vs_shoulder_separation": separation_score(df, col, BENCH, SHOULDER)})

ranking = pd.DataFrame(rows).sort_values("chest_vs_shoulder_separation", ascending=False)
ranking.reset_index(drop=True)